In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the importance-magnitude CSV files
input_folder = Path(".")

# Match files such as:
filename_pattern = re.compile(
    r"^importance_magnitude_mlp_RfxCas13a_validation_(\d+)\.csv$"
)

# Detect matching files
matching_files = []

for file_path in input_folder.glob(
    "importance_magnitude_mlp_RfxCas13a_validation_*.csv"
):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

# Sort by model number
matching_files.sort(key=lambda x: x[0])

print(f"Detected {len(matching_files)} importance-magnitude files:")

for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        "No matching importance-magnitude CSV files were found."
    )

if len(matching_files) != 10:
    print(
        f"\nWarning: Expected 10 files, but detected "
        f"{len(matching_files)} files."
    )


# Read and combine all files
all_importance_tables = []

for model_number, file_path in matching_files:

    df = pd.read_csv(file_path)

    # Check that the expected columns exist
    required_columns = {"Feature", "Importance Magnitude"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{file_path.name} does not contain the required columns: "
            f"'Feature' and 'Importance Magnitude'."
        )

    # Keep only the required columns
    df = df[["Feature", "Importance Magnitude"]].copy()

    # Convert importance values to numeric
    df["Importance Magnitude"] = pd.to_numeric(
        df["Importance Magnitude"],
        errors="coerce"
    )

    # Add model number for tracking
    df["Model Number"] = model_number

    all_importance_tables.append(df)

# Combine all model tables
combined_df = pd.concat(
    all_importance_tables,
    ignore_index=True
)

# Calculate average importance magnitude for each feature
average_importance = (
    combined_df
    .groupby("Feature", as_index=False)
    .agg(
        Average_Importance_Magnitude=(
            "Importance Magnitude",
            "mean"
        ),
        Number_of_Models=(
            "Importance Magnitude",
            "count"
        )
    )
)

# Sort from highest to lowest average importance
average_importance = average_importance.sort_values(
    by="Average_Importance_Magnitude",
    ascending=False,
    ignore_index=True
)

# Save the final table
output_file = (
    input_folder /
    "average_importance_magnitude_mlp_RfxCas13a_validation.csv"
)

average_importance.to_csv(output_file, index=False)

print(f"\nSaved: {output_file.name}")
print("\nAverage importance magnitude table:")
print(average_importance)

Detected 10 importance-magnitude files:
Model 9: importance_magnitude_mlp_RfxCas13a_validation_9.csv
Model 22: importance_magnitude_mlp_RfxCas13a_validation_22.csv
Model 41: importance_magnitude_mlp_RfxCas13a_validation_41.csv
Model 42: importance_magnitude_mlp_RfxCas13a_validation_42.csv
Model 56: importance_magnitude_mlp_RfxCas13a_validation_56.csv
Model 73: importance_magnitude_mlp_RfxCas13a_validation_73.csv
Model 76: importance_magnitude_mlp_RfxCas13a_validation_76.csv
Model 83: importance_magnitude_mlp_RfxCas13a_validation_83.csv
Model 88: importance_magnitude_mlp_RfxCas13a_validation_88.csv
Model 98: importance_magnitude_mlp_RfxCas13a_validation_98.csv

Saved: average_importance_magnitude_mlp_RfxCas13a_validation.csv

Average importance magnitude table:
                                              Feature  \
0                                        crRNA spacer   
1         bh_maximal consecutive paired bases (crRNA)   
2                              bh_5' overhang (crRNA)   
3

In [1]:
import pandas as pd

# Input and output files
input_file = "average_importance_magnitude_mlp_RfxCas13a_validation.csv"
output_file = "reordered_average_importance_magnitude_mlp_RfxCas13a_validation.csv"

# Desired feature order
desired_order = [
    "crRNA spacer",
    "bh_maximal consecutive paired bases (crRNA)",
    "bh_maximal consecutive unpaired bases (crRNA)",
    "bh_5' overhang (crRNA)",
    "bh_3' overhang (crRNA)",
    "bh_5' stem (crRNA)",
    "bh_3' stem (crRNA)",
    "ssRNA target",
    "bh_maximal consecutive paired bases (ssRNA)",
    "bh_maximal consecutive unpaired bases (ssRNA)",
    "bh_5' overhang (ssRNA)",
    "bh_3' overhang (ssRNA)",
    "bh_5' stem (ssRNA)",
    "bh_3' stem (ssRNA)",
    "crRNA-target duplex",
    "ah_maximal consecutive paired bases (crRNA in duplex)",
    "ah_maximal consecutive unpaired bases (crRNA in duplex)",
    "ah_5' overhang (crRNA in duplex)",
    "ah_3' overhang (crRNA in duplex)",
    "ah_5' stem (crRNA in duplex)",
    "ah_3' stem (crRNA in duplex)",
    "PFS-proximal region",
    "central region",
    "PFS-distal region"
]

# Read the CSV
df = pd.read_csv(input_file)

# Make the Feature column a categorical with the desired order
df["Feature"] = pd.Categorical(
    df["Feature"],
    categories=desired_order,
    ordered=True
)

# Sort according to the specified order
df = df.sort_values("Feature").reset_index(drop=True)

# Save the reordered CSV
df.to_csv(output_file, index=False)

print(f"Reordered file saved as '{output_file}'.")

Reordered file saved as 'reordered_average_importance_magnitude_mlp_RfxCas13a_validation.csv'.
